# 02 - Raw to Bronze Load
## FoodLens Databricks Pipeline
### Purpose: Read CSVs from Raw Volume → Write Parquet to Staging Volume → Load Bronze Delta Tables

In [0]:
%run "./Util"

## Step 1: Imports and Helper Functions

In [0]:
from datetime import datetime
from pyspark.sql import functions as F

def file_exists(path):
    try:
        dbutils.fs.ls(path)
        return True
    except Exception:
        return False

def read_csv(path):
    return (
        spark.read
        .format("csv")
        .option("header",                  "true")
        .option("inferSchema",             "false")
        .option("multiLine",               "true")
        .option("quote",                   '"')
        .option("escape",                  '"')
        .option("ignoreLeadingWhiteSpace", "true")
        .option("ignoreTrailingWhiteSpace","true")
        .option("encoding",                "UTF-8")
        .load(path)
    )

def write_parquet_staging(df, stg_path, city, batch_id, RUN_TS):
    df_out = (
        df
        .withColumn("_batch_id",    F.lit(batch_id))
        .withColumn("_source_city", F.lit(city))
        .withColumn("_ingested_at", F.lit(RUN_TS))
    )
    (df_out
        .write
        .format("parquet")
        .mode("overwrite")
        .option("compression", "snappy")
        .save(stg_path)
    )

    if file_exists(stg_path):
        target_df = spark.read.parquet(stg_path)
        target_row_count = target_df.count()
        print(f"{city} - {target_row_count:,} rows written to staging")
    else:
        print(f"ERROR: {city} Staging write failed!!!")
        return 0, None

    return target_row_count, target_df

print("✅ Helper functions loaded!")

✅ Helper functions loaded!


## Step 2: Read Active Tables from Parent Metadata
Pipeline is metadata-driven — reads pipeline_control to get list of tables to process.

In [0]:
active_tables_df = spark.sql(f"""
    SELECT DISTINCT table_name
    FROM {CATALOG}.{META_SCHEMA}.{PARENT_META_TABLE}
    WHERE active_flag = 'Y'
""")

active_tables = [row["table_name"] for row in active_tables_df.collect()]
print(f"Active files to process: {active_tables}")
active_tables_df.show()

Active files to process: ['Chicago', 'Dallas']
+----------+
|table_name|
+----------+
|   Chicago|
|    Dallas|
+----------+



## Step 3: Extract from Source and Write to Raw + Bronze
For each active table:
- Confirm file exists in Raw Volume
- Read CSV from Raw Volume
- Write Parquet to Staging Volume with timestamped path
- Verify source vs target row count
- Log results to child metadata table
- Write Delta table to Bronze layer

In [0]:
for table_name in active_tables:
    try:
        print(f"\nProcessing {table_name} at {datetime.now()}")

        # Step 0: Confirm file exists
        if not file_exists(LANDING_FILE_MAP.get(table_name)):
            print(f"WARN: File not found: {LANDING_FILE_MAP.get(table_name)}")
            spark.sql(f"""
                INSERT INTO {CATALOG}.{META_SCHEMA}.{CHILED_META_TABLE} VALUES (
                    '{table_name}',
                    '{datetime.now()}',
                    'SKIPPED',
                    0, 0, 'N/A',
                    current_date()
                )
            """)
            continue

        # Step 1: Read CSV from Raw Volume
        staging_df = read_csv(LANDING_FILE_MAP.get(table_name))
        source_row_count = staging_df.count()

        # Step 2: Build dynamic timestamped file path
        RUN_TS      = datetime.now()
        RUN_TS_STR  = RUN_TS.strftime("%Y%m%d_%H%M%S")
        RUN_DATE    = RUN_TS.strftime("%Y/%m/%d")
        BATCH_ID    = f"BATCH_{RUN_TS_STR}"

        STG_FILE_NAME = f"{FILE_NAME_MAP.get(table_name).lower() + '_' + RUN_TS_STR}.parquet"
        STG_FILE_PATH = f"{STAGING_VOLUME_PATH}/{RUN_DATE}/{STG_FILE_NAME}"

        # Step 3: Write Parquet to Staging Volume
        target_row_count, target_df = write_parquet_staging(
            staging_df, STG_FILE_PATH, table_name, BATCH_ID, RUN_TS
        )

        # Step 4: Determine status
        status = "SUCCESS" if source_row_count == target_row_count else "FAILED"

        # Step 5: Log to child metadata
        spark.sql(f"""
            INSERT INTO {CATALOG}.{META_SCHEMA}.{CHILED_META_TABLE} VALUES (
                '{table_name}',
                '{RUN_TS}',
                '{status}',
                {source_row_count},
                {target_row_count},
                '{STG_FILE_PATH}',
                current_date()
            )
        """)

        if status == "SUCCESS":
            print(f"Written to  : {STG_FILE_PATH}")
            print(f"{table_name} | {status} | Source: {source_row_count} | Target: {target_row_count} | Batch ID: {BATCH_ID}")

            # Step 6: Rename columns (remove spaces)
            target_df = target_df.toDF(*[c.replace(" ", "_") for c in target_df.columns])

            # Step 7: Add audit columns and write to Bronze Delta
            df_bronze = (
                target_df
                .withColumn("_bronze_loaded_at", F.lit(datetime.now()))
                .withColumn("_batch_id",         F.lit(BATCH_ID))
            )
            (
                df_bronze.write
                .format("delta")
                .mode("overwrite")
                .option("overwriteSchema", "true")
                .saveAsTable(f"{CATALOG}.{BRONZE_SCHEMA}.{BRONZE_TABLE_MAP.get(table_name)}")
            )
            print(f"\nWritten to  : {CATALOG}.{BRONZE_SCHEMA}.{BRONZE_TABLE_MAP.get(table_name)}")
        else:
            print(f"\nERROR: {table_name} | {status} | Source: {source_row_count} | Target: {target_row_count}")

        print("\n" + "-" * 80)

    except Exception as e:
        import traceback
        print(f"{table_name} FAILED: {str(e)}")
        print(traceback.format_exc())
        spark.sql(f"""
            INSERT INTO {CATALOG}.{META_SCHEMA}.{CHILED_META_TABLE} VALUES (
                '{table_name}',
                '{datetime.now()}',
                'FAILED',
                0, 0, 'N/A',
                current_date()
            )
        """)

print("\n✅ Raw load completed!")


Processing Chicago at 2026-04-21 20:18:56.357888
Chicago - 308,161 rows written to staging
Written to  : /Volumes/foodlens/bronze_zone/vol_stg_files/2026/04/21/chicago_restaurant_and_food_establishment_inspections_20260421_201905.parquet
Chicago | SUCCESS | Source: 308161 | Target: 308161 | Batch ID: BATCH_20260421_201905

Written to  : foodlens.bronze_zone.bronze_chicago

--------------------------------------------------------------------------------

Processing Dallas at 2026-04-21 20:19:24.010088
Dallas - 78,984 rows written to staging
Written to  : /Volumes/foodlens/bronze_zone/vol_stg_files/2026/04/21/dallas_restaurant_and_food_establishment_inspections_20260421_201928.parquet
Dallas | SUCCESS | Source: 78984 | Target: 78984 | Batch ID: BATCH_20260421_201928

Written to  : foodlens.bronze_zone.bronze_dallas

--------------------------------------------------------------------------------

✅ Raw load completed!


## Step 4: Validate Bronze Tables

In [0]:
print("=" * 50)
print("BRONZE LOAD SUMMARY")
print("=" * 50)

# Check child metadata log
spark.sql(f"""
    SELECT table_name, status, source_row_count, target_row_count, execution_time
    FROM {CATALOG}.{META_SCHEMA}.{CHILED_META_TABLE}
    ORDER BY execution_time DESC
""").show(truncate=False)

# Spot check Bronze tables
print("Bronze Dallas row count:")
spark.sql(f"SELECT COUNT(*) as row_count FROM {CATALOG}.{BRONZE_SCHEMA}.{BRONZE_DALLAS_TABLE}").show()

print("Bronze Chicago row count:")
spark.sql(f"SELECT COUNT(*) as row_count FROM {CATALOG}.{BRONZE_SCHEMA}.{BRONZE_CHICAGO_TABLE}").show()

print("✅ Bronze validation complete!")

BRONZE LOAD SUMMARY
+----------+-------+----------------+----------------+--------------------------+
|table_name|status |source_row_count|target_row_count|execution_time            |
+----------+-------+----------------+----------------+--------------------------+
|Dallas    |SUCCESS|78984           |78984           |2026-04-21 20:19:28.020174|
|Chicago   |SUCCESS|308161          |308161          |2026-04-21 20:19:05.039593|
|Dallas    |SUCCESS|78984           |78984           |2026-04-21 15:05:28.300225|
|Chicago   |SUCCESS|308161          |308161          |2026-04-21 15:04:56.309241|
|Dallas    |SUCCESS|78984           |78984           |2026-04-21 14:36:58.860987|
|Chicago   |SUCCESS|308161          |308161          |2026-04-21 14:36:37.354643|
+----------+-------+----------------+----------------+--------------------------+

Bronze Dallas row count:
+---------+
|row_count|
+---------+
|    78984|
+---------+

Bronze Chicago row count:
+---------+
|row_count|
+---------+
|   308161|